# Massive Activation Perturbation Test: Token Identity vs. Positional Structure

**Hypothesis under test:** Sun et al. (COLM 2024) claim massive activations (MAs) are
*input-agnostic* — their values stay roughly constant regardless of input content. We test
a stronger, more specific version of this claim: is the MA at position 0 driven by the
**semantic identity** of the token occupying that position, or is it a **structural/positional**
effect that persists even when the position-0 token embedding is replaced by a random
direction of the same norm?

**Design**
1. Generate N random nonsensical sentences (real vocab tokens, no semantic structure), no BOS.
2. Run a baseline forward pass, confirm MAs replicate at position 0 / known channels
   (~788, 1384, 4062 for Llama-3.1-8B), consistent with prior notebook (`01f_*`).
3. Perturb the **position-0 input embedding** with a uniformly random direction on the
   hypersphere of the same norm (full direction randomization, exact norm preservation).
4. Re-run forward pass, compare MA magnitude/location pre- vs. post-perturbation.
5. Control: repeat the identical perturbation at a mid-sequence position instead of
   position 0, to confirm any effect is position-0-specific rather than generic.

**Interpretation:** if MA magnitude survives full direction randomization at fixed norm,
this is strong evidence MAs are structural/positional (input-agnostic in the strong sense),
supporting the confound-check framing (MAs as a control analysis, not a causal driver of
Jacobian spike behavior) rather than the reverse.

No BOS token is prepended anywhere in this notebook — position 0 is the first random
content token, consistent with the `01f_block_jacobian_svd_sweep.ipynb` setup.


## 1. Setup

In [1]:
import os
import random
import json
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm

MODEL_PATH = "/home/samuel/research/llmattacks/llm-attacks/DIR/Llama-3.1-8B-Instruct"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Device: {DEVICE}, dtype: {DTYPE}")


Device: cuda, dtype: torch.bfloat16


In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=DTYPE,
    device_map=DEVICE,
)
model.eval()

VOCAB_SIZE = model.get_input_embeddings().weight.shape[0]
D_MODEL = model.get_input_embeddings().weight.shape[1]
N_LAYERS = model.config.num_hidden_layers

OUTPUT_DIR = "activations_output"  # set to whatever path you want

print(f"Vocab size: {VOCAB_SIZE}, d_model: {D_MODEL}, n_layers: {N_LAYERS}")


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Vocab size: 128256, d_model: 4096, n_layers: 32


## 2. Config

Known massive-activation channels and the reference position, carried over from
`01f_block_jacobian_svd_sweep.ipynb`. Adjust if the earlier notebook's findings change.

In [3]:
N_SAMPLES = 10000        # number of random nonsensical sentences
SEQ_LEN = 2  # inclusive, tokens per sentence
PERTURB_POS = 0          # position 0 = first content token (no BOS)

KNOWN_MA_CHANNELS = [788, 1384, 4062]   # from prior notebook, replicated at position 0
MA_RATIO_THRESHOLD = 1000               # "genuine" massive activation threshold (Sun et al. convention)


## 3. Sample random nonsensical sentences

Token IDs are drawn uniformly from the vocabulary, excluding special/reserved tokens
(Llama-3.1's tokenizer reserves a large block of `<|reserved_special_token_*|>` and other
control tokens that would pollute both the "nonsensical sentence" semantics and the
activation statistics).

In [4]:
# def get_excluded_token_ids(tokenizer):
#     excluded = set(tokenizer.all_special_ids)
#     # Sweep the vocab once and flag anything that decodes to a reserved/control-looking
#     # token, as a fallback in case all_special_ids doesn't capture every reserved slot.
#     for tid in range(tokenizer.vocab_size, len(tokenizer.get_vocab())):
#         excluded.add(tid)  # anything beyond base vocab_size is typically added/special
#     return excluded
# EXCLUDED_IDS = get_excluded_token_ids(tokenizer)

def get_excluded_token_ids(model, threshold=1e-6):
    """Return the set of token IDs whose embedding vector has (numerically)
    zero norm (Dr. Liu's reserved/untrained tokens, ~266 tokens with norm
    ~1e-21), to be excluded from sampling."""
    embed_norms = model.model.embed_tokens.weight.norm(dim=-1)
    excluded = set(torch.nonzero(embed_norms < threshold, as_tuple=True)[0].tolist())
    return excluded

EXCLUDED_IDS = get_excluded_token_ids(model)
print(len(EXCLUDED_IDS))
VALID_IDS = np.array([i for i in range(VOCAB_SIZE) if i not in EXCLUDED_IDS])
print(f"Excluded {len(EXCLUDED_IDS)} tokens with zero embeddings; {len(VALID_IDS)} valid ids remain")


266
Excluded 266 tokens with zero embeddings; 127990 valid ids remain


In [5]:
def sample_random_sentence(rng):
    token_ids = rng.choice(VOCAB_SIZE, size=SEQ_LEN, replace=True).tolist()
    return token_ids

rng = np.random.default_rng(SEED)
samples = [sample_random_sentence(rng) for _ in range(N_SAMPLES)]
print(len(samples))

10000


In [6]:
# # Print decoded token lists for every sample as a sanity check.
# print("Decoded samples (sanity check):")
# for i, ids in enumerate(samples):
#     tokens = tokenizer.convert_ids_to_tokens(ids)
#     print(f"[{i:03d}] ids={ids}")
#     print(f"      tokens={tokens}")

## 5. Norm-preserving perturbation

For the token at the target position, replace its input embedding with a uniformly random
point on the hypersphere of the same norm:

$$e' = \|e\| \cdot \frac{n}{\|n\|}, \quad n \sim \mathcal{N}(0, I_d)$$

This is the full-randomization extreme case: in ~4096-d space, `e'` has expected cosine
similarity ≈ 0 with the original embedding `e`, while `||e'|| = ||e||` exactly.

In [7]:
# Representative embedding norm: median over the full vocabulary's row norms.
# The specific token/vector attaining it is irrelevant — this is just a robust
# (outlier-resistant) scalar summary of "how big is a typical embedding vector".
embedding_matrix = model.get_input_embeddings().weight.detach().float()
embedding_row_norms = embedding_matrix[VALID_IDS].norm(dim=1)
MEDIAN_EMBEDDING_NORM = embedding_row_norms.median().item()

print(f"Embedding row norms — min={embedding_row_norms.min().item():.4f}  "
      f"median={MEDIAN_EMBEDDING_NORM:.4f}  max={embedding_row_norms.max().item():.4f}")


Embedding row norms — min=0.0007  median=0.6851  max=0.9313


In [8]:
def perturb_embedding_random_direction(embedding_row, generator, target_norm=None):
    """embedding_row: [d_model] tensor. Returns a new tensor with the given target_norm
    (default: the row's own norm) and a random direction.
    
    target_norm: scalar to scale the random unit vector to. Pass MEDIAN_EMBEDDING_NORM
    to scale to the vocabulary-wide median embedding norm instead of this token's own norm.
    """
    d = embedding_row.shape[0]
    noise = torch.randn(d, generator=generator, device=embedding_row.device, dtype=torch.float32)
    unit_noise = noise / noise.norm()

    if target_norm is None:
        target_norm = embedding_row.float().norm()

    new_row = unit_noise * target_norm
    return new_row.to(embedding_row.dtype)


def build_inputs_embeds_with_perturbation(input_ids, perturb_positions, torch_gen):
    """Return inputs_embeds [1, seq, d_model] with the tokens at perturb_positions replaced
    by random-direction, same-norm embeddings.

    perturb_positions: list/tuple of int positions to perturb (each gets an independent
    random direction, drawn from torch_gen).
    """
    embed_layer = model.get_input_embeddings()
    with torch.no_grad():
        base_embeds = embed_layer(input_ids).clone()  # [1, seq, d_model]
        for pos in perturb_positions:
            original_row = base_embeds[0, pos, :]
            perturbed_row = perturb_embedding_random_direction(original_row, torch_gen, target_norm=MEDIAN_EMBEDDING_NORM)
            base_embeds[0, pos, :] = perturbed_row
    return base_embeds


def build_inputs_embeds_no_perturbation(input_ids):
    """Return inputs_embeds [1, seq, d_model] with the token at perturb_position replaced
    by a random-direction, same-norm embedding."""
    embed_layer = model.get_input_embeddings()
    with torch.no_grad():
        base_embeds = embed_layer(input_ids).clone()  # [1, seq, d_model]
    return base_embeds


# Fresh, non-reproducible seed for the perturbation generator (independent of the
# fixed SEED above, which only controls sentence sampling) — each run picks a
# different random direction.
PERTURB_SEED = int.from_bytes(os.urandom(4), "big")
torch_gen = torch.Generator(device=DEVICE)
torch_gen.manual_seed(PERTURB_SEED)
print(f"torch_gen seeded with PERTURB_SEED={PERTURB_SEED}")

torch_gen seeded with PERTURB_SEED=2631780342


## 6. Perturbation experiment

In [9]:
@torch.no_grad()
def run_forward(input_ids=None, inputs_embeds=None):
    """Run a forward pass from either input_ids or inputs_embeds, return hidden_states tuple."""
    if input_ids is not None:
        attention_mask = torch.ones_like(input_ids)
        out = model(input_ids=input_ids, attention_mask=attention_mask,
                     output_hidden_states=True, use_cache=False)
    else:
        attention_mask = torch.ones(inputs_embeds.shape[:2], dtype=torch.long, device=inputs_embeds.device)
        out = model(inputs_embeds=inputs_embeds, attention_mask=attention_mask,
                     output_hidden_states=True, use_cache=False)
    # hidden_states: tuple of (n_layers + 1) tensors, each [batch, seq, d_model]
    return out.hidden_states

In [10]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
# --- register MLP pre-hooks once, outside the loop ---
captured = {}

def make_pre_mlp_hook(name):
    def hook(module, inputs):
        captured[name] = inputs[0].detach()
    return hook

h0_handle = model.model.layers[0].mlp.register_forward_pre_hook(make_pre_mlp_hook("H1_prime"))
h1_handle = model.model.layers[1].mlp.register_forward_pre_hook(make_pre_mlp_hook("H2_prime"))

perturb_positions = [0, 1]
CAPTURE_POS = 0
rows = []  # will hold 5 rows per sample: [H0, H1', H1, H2', H2]
# for i, ids in enumerate(samples):

for i, ids in enumerate(tqdm(samples, desc="Propagating samples", total=len(samples))):
    input_ids = torch.tensor([ids], device=DEVICE)
    # # Random embeddings
    # inputs_embeds = build_inputs_embeds_with_perturbation(input_ids, perturb_positions, torch_gen)
    # hidden_states = run_forward(inputs_embeds=inputs_embeds)
    # # Random embeddings
    hidden_states = run_forward(input_ids=input_ids)

    H0 = hidden_states[0][0, CAPTURE_POS, :].float()
    H1 = hidden_states[1][0, CAPTURE_POS, :].float()
    H2 = hidden_states[2][0, CAPTURE_POS, :].float()
    H1_prime = captured["H1_prime"][0, CAPTURE_POS, :].float()
    H2_prime = captured["H2_prime"][0, CAPTURE_POS, :].float()

    rows.append(H0.cpu().numpy())
    rows.append(H1_prime.cpu().numpy())
    rows.append(H1.cpu().numpy())
    rows.append(H2_prime.cpu().numpy())
    rows.append(H2.cpu().numpy())

h0_handle.remove()
h1_handle.remove()

result = np.stack(rows, axis=0)   # (5 * n_samples, d_model) -> (500, 4096) for 100 samples
out_path = os.path.join(OUTPUT_DIR, "activations_h0_h1p_h1_h2p_h2.npy")
np.save(out_path, result)
print(result.shape)

Propagating samples: 100%|██████████| 10000/10000 [04:50<00:00, 34.45it/s]


(50000, 4096)


## 7. Harmful and Harmless Prompts - Experiment

In [11]:
N_HARMFUL = 500
N_HARMLESS = 500
POSITIONS = [0, 1, 2]
HIDDEN_DIM = D_MODEL

In [12]:
DATA_DIR = "./phase1/data/saladbench_splits"
HARMFUL_JSON_PATH = f"{DATA_DIR}/harmful_val.json"    # each: [{"instruction": "..."}, ...]
HARMLESS_JSON_PATH = f"{DATA_DIR}/harmless_val.json"  # each: [{"instruction": "..."}, ...]
OUT_MATRIX = "harmful_harmless_dataset_activations.npy"
OUT_META = "harmful_harmless_dataset_metadata.csv"

ACTIVATION_NAMES = [
    "H_0", "H_1p", "H_1", "H_2p", "H_2",
    "H_30", "H_31p", "H_31", "H_32p", "H_32", "H_final",
]
N_ACT = len(ACTIVATION_NAMES)  # 11


In [13]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

n_layers = len(model.model.layers)
assert n_layers == 32, f"Expected 32 decoder layers, found {n_layers} -- check LAYER indices below."

# ------------------------------------------------------------------
# 2. Load harmful / harmless prompts from local JSON
#    Expected format: [{"instruction": "..."}, {"instruction": "..."}, ...]
# ------------------------------------------------------------------
def load_prompts_from_json(path, n, seed=SEED):
    with open(path, "r") as f:
        data = json.load(f)
    instructions = [ex["instruction"] for ex in data]
    if len(instructions) < n:
        raise ValueError(f"{path} has only {len(instructions)} prompts, need {n}")
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(instructions), size=n, replace=False)
    return [instructions[i] for i in idx]

harmful_prompts = load_prompts_from_json(HARMFUL_JSON_PATH, N_HARMFUL)
harmless_prompts = load_prompts_from_json(HARMLESS_JSON_PATH, N_HARMLESS)

prompts = harmful_prompts + harmless_prompts
labels = ["harmful"] * len(harmful_prompts) + ["harmless"] * len(harmless_prompts)
print(f"Loaded {len(harmful_prompts)} harmful + {len(harmless_prompts)} harmless = {len(prompts)} prompts")

# ------------------------------------------------------------------
# 3. Hooks
# ------------------------------------------------------------------
captured = {}

def make_pre_mlp_hook(name):
    def hook(module, inputs):
        captured[name] = inputs[0].detach()
    return hook

def make_post_layer_hook(name):
    def hook(module, inputs, output):
        out = output[0] if isinstance(output, tuple) else output
        captured[name] = out.detach()
    return hook

handles = [
    model.model.layers[0].mlp.register_forward_pre_hook(make_pre_mlp_hook("H_1p")),
    model.model.layers[1].mlp.register_forward_pre_hook(make_pre_mlp_hook("H_2p")),
    model.model.layers[30].mlp.register_forward_pre_hook(make_pre_mlp_hook("H_31p")),
    model.model.layers[31].mlp.register_forward_pre_hook(make_pre_mlp_hook("H_32p")),
    model.model.layers[31].register_forward_hook(make_post_layer_hook("H_32_raw")),
]

# ------------------------------------------------------------------
# 4. Tokenize raw (no chat template, no special tokens/BOS)
# ------------------------------------------------------------------
tokenized = [tokenizer(p, add_special_tokens=False)["input_ids"] for p in prompts]

valid_idx = [i for i, ids in enumerate(tokenized) if len(ids) >= 3]
skipped = len(tokenized) - len(valid_idx)
if skipped:
    print(f"Skipping {skipped} prompt(s) with fewer than 3 tokens.")

tokenized = [tokenized[i] for i in valid_idx]
labels = [labels[i] for i in valid_idx]
prompts = [prompts[i] for i in valid_idx]
n_samples = len(tokenized)

pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

# ------------------------------------------------------------------
# 5. Forward pass, batched, right-padded
# ------------------------------------------------------------------
result = np.zeros((n_samples * len(POSITIONS) * N_ACT, HIDDEN_DIM), dtype=np.float32)
meta_rows = []
row_ptr = 0

@torch.no_grad()
def run_batch(id_batch):
    max_len = max(len(ids) for ids in id_batch)
    input_ids = torch.full((len(id_batch), max_len), pad_id, dtype=torch.long, device=DEVICE)
    attention_mask = torch.zeros((len(id_batch), max_len), dtype=torch.long, device=DEVICE)
    for i, ids in enumerate(id_batch):
        input_ids[i, :len(ids)] = torch.tensor(ids, device=DEVICE)
        attention_mask[i, :len(ids)] = 1

    out = model(input_ids=input_ids, attention_mask=attention_mask,
                output_hidden_states=True, use_cache=False)
    hs = out.hidden_states  # 33 tensors, each (bs, seq, 4096)
    return hs

for sample_i in tqdm(range(n_samples), desc="Dataset C samples"):
    id_batch = [tokenized[sample_i]]

    hs = run_batch(id_batch)

    acts = {
        "H_0": hs[0],
        "H_1p": captured["H_1p"],
        "H_1": hs[1],
        "H_2p": captured["H_2p"],
        "H_2": hs[2],
        "H_30": hs[30],
        "H_31p": captured["H_31p"],
        "H_31": hs[31],
        "H_32p": captured["H_32p"],
        "H_32": captured["H_32_raw"],   # raw, pre-final-norm
        "H_final": hs[32],              # post-final-norm
    }

    for pos in POSITIONS:
        for act_name in ACTIVATION_NAMES:
            vec = acts[act_name][0, pos, :].float().cpu().numpy()
            result[row_ptr] = vec
            meta_rows.append({
                "row_index": row_ptr,
                "sample_id": sample_i,
                "label": labels[sample_i],
                "position": pos,
                "activation_name": act_name,
            })
            row_ptr += 1

for h in handles:
    h.remove()

assert row_ptr == result.shape[0]

# ------------------------------------------------------------------
# 6. Save
# ------------------------------------------------------------------
out_matrix_path = os.path.join(OUTPUT_DIR, OUT_MATRIX)
out_meta_path = os.path.join(OUTPUT_DIR, OUT_META)

np.save(out_matrix_path, result)
pd.DataFrame(meta_rows).to_csv(out_meta_path, index=False)
print(f"Saved matrix {result.shape} -> {out_matrix_path}")
print(f"Saved metadata ({len(meta_rows)} rows) -> {out_meta_path}")


# ------------------------------------------------------------------
# 7. Sanity check
# ------------------------------------------------------------------
print("\nSanity check -- sample 0, position 0, row norms:")
for k, name in enumerate(ACTIVATION_NAMES):
    print(f"  {name}: {np.linalg.norm(result[k]):.4f}")



Loaded 500 harmful + 500 harmless = 1000 prompts


Dataset C samples: 100%|██████████| 1000/1000 [00:31<00:00, 31.99it/s]


Saved matrix (33000, 4096) -> activations_output/harmful_harmless_dataset_activations.npy
Saved metadata (33000 rows) -> activations_output/harmful_harmless_dataset_metadata.csv

Sanity check -- sample 0, position 0, row norms:
  H_0: 0.7051
  H_1p: 8.0174
  H_1: 1.8424
  H_2p: 11.7189
  H_2: 509.0291
  H_30: 513.2711
  H_31p: 22.6863
  H_31: 507.8592
  H_32p: 22.8161
  H_32: 52.0938
  H_final: 138.0736


In [14]:
# arr = np.load(os.path.join(OUTPUT_DIR, "activations_h0_h1p_h1_h2p_h2.npy"))
arr = np.load(os.path.join(OUTPUT_DIR, "harmful_harmless_dataset_activations.npy"))
print(arr.shape)
print(arr.dtype)         # should print: float32
print(f"{arr.nbytes:,} bytes  ({arr.nbytes / (1024**2):.1f} MiB)")

(33000, 4096)
float32
540,672,000 bytes  (515.6 MiB)


In [15]:
# perturb_positions = [0,1]
# records = []

# for i, ids in enumerate(samples):
#     input_ids = torch.tensor([ids], device=DEVICE)
#     inputs_embeds = build_inputs_embeds_with_perturbation(input_ids, perturb_positions, torch_gen)
    
#     hidden_states = run_forward(inputs_embeds=inputs_embeds)

#     seq_len = input_ids.shape[1]
#     for layer, hs in enumerate(hidden_states):
#         hs_0 = hs[0].float()                              # [seq_len, d_model]
#         max_abs_per_pos = hs_0.abs().max(dim=-1).values    # [seq_len]
#         argmax_per_pos = hs_0.abs().argmax(dim=-1)         # [seq_len]  <-- channel index of the max
#         norm_per_pos = hs_0.norm(dim=-1)                   # [seq_len]
#         for pos in range(seq_len):
#             records.append({
#                 "sample": i,
#                 "layer": layer,
#                 "position": pos,
#                 "max_activation": max_abs_per_pos[pos].item(),
#                 "max_activation_channel": argmax_per_pos[pos].item(),
#                 "activation_norm": norm_per_pos[pos].item(),
#             })


## Additional Codes

In [16]:
# # Saving the results and statistics
# os.makedirs(OUTPUT_DIR, exist_ok=True)

# per_position_df = pd.DataFrame(records)

# layer_sum_df = (
#     per_position_df.groupby(["sample", "layer"])["max_activation"]
#     .sum()
#     .reset_index()
#     .rename(columns={"max_activation": "sum_max_activation"})
# )

# PER_POSITION_CSV = os.path.join(OUTPUT_DIR, f"ma_random_test_per_position_{N_SAMPLES}.csv")
# LAYER_SUM_CSV = os.path.join(OUTPUT_DIR, f"ma_random_test_layer_sum_{N_SAMPLES}.csv")

# per_position_df.to_csv(PER_POSITION_CSV, index=False)
# layer_sum_df.to_csv(LAYER_SUM_CSV, index=False)

# print(f"Saved per-position max activations -> {PER_POSITION_CSV} ({len(per_position_df)} rows)")
# print(f"Saved per-layer sum of max activations -> {LAYER_SUM_CSV} ({len(layer_sum_df)} rows)")

In [17]:
# per_position_df = pd.DataFrame(records)

# layer_sum_df = (
#     per_position_df.groupby(["sample", "layer"])["max_activation"]
#     .sum()
#     .reset_index()
#     .rename(columns={"max_activation": "sum_max_activation"})
# )

# PER_POSITION_CSV = "ma_random_test_per_position.csv"
# LAYER_SUM_CSV = "ma_random_test_layer_sum.csv"

# per_position_df.to_csv(PER_POSITION_CSV, index=False)
# layer_sum_df.to_csv(LAYER_SUM_CSV, index=False)

# print(f"Saved per-position max activations -> {PER_POSITION_CSV} ({len(per_position_df)} rows)")
# print(f"Saved per-layer sum of max activations -> {LAYER_SUM_CSV} ({len(layer_sum_df)} rows)")
